# Evaluation

Time-based offline evaluation for the baseline and hybrid recommenders at K = 5, 10, and 20.

In [1]:
import importlib.util
import sys
from pathlib import Path

import pandas as pd

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "baseline-model").exists() and (candidate / "hybrid-model").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repository root")
BASELINE_BACKEND = REPO_ROOT / "baseline-model" / "backend"
if str(BASELINE_BACKEND) not in sys.path:
    sys.path.append(str(BASELINE_BACKEND))

baseline_spec = importlib.util.spec_from_file_location(
    "baseline_model_recommender",
    BASELINE_BACKEND / "recommender" / "baseline.py",
)
baseline_module = importlib.util.module_from_spec(baseline_spec)
baseline_spec.loader.exec_module(baseline_module)
BaselineRecommender = baseline_module.BaselineRecommender

from recommender.evaluator import Evaluator
from recommender.hybrid import HybridRecommender
from recommender.utils import load_data

products, users, interactions = load_data()
K_VALUES = [5, 10, 20]

print(f"Products: {len(products):,}")
print(f"Users: {len(users):,}")
print(f"Interactions: {len(interactions):,}")
print("Test period: interactions from 2025-01-01 onward")

Products: 500
Users: 300
Interactions: 6,194
Test period: interactions from 2025-01-01 onward


## Hybrid Model Evaluation

In [2]:
hybrid_model = HybridRecommender().fit(products, users, interactions)
evaluator = Evaluator()
hybrid_results = evaluator.evaluate(hybrid_model, interactions, products, K_VALUES)
evaluator.print_report(hybrid_results)

K | Precision | Recall | NDCG | Coverage | Diversity | Users
--|-----------|--------|------|----------|-----------|------
5 | 0.059 | 0.044 | 0.053 | 0.638 | 0.301 | 299
10 | 0.054 | 0.082 | 0.065 | 0.894 | 0.353 | 299
20 | 0.043 | 0.133 | 0.087 | 0.936 | 0.408 | 299


## Baseline Model Evaluation

In [3]:
baseline_model = BaselineRecommender().fit(products, interactions)
baseline_results = evaluator.evaluate(baseline_model, interactions, products, K_VALUES)
evaluator.print_report(baseline_results)

K | Precision | Recall | NDCG | Coverage | Diversity | Users
--|-----------|--------|------|----------|-----------|------
5 | 0.019 | 0.015 | 0.021 | 0.010 | 0.900 | 299
10 | 0.017 | 0.027 | 0.025 | 0.020 | 0.800 | 299
20 | 0.018 | 0.060 | 0.039 | 0.040 | 0.858 | 299


## Baseline vs Hybrid Comparison

In [4]:
def flatten_results(model_name: str, results: dict) -> list[dict]:
    rows = []
    for k in K_VALUES:
        metrics = results[k]
        rows.append(
            {
                "Model": model_name,
                "K": k,
                "Precision": metrics["precision"]["mean"],
                "Recall": metrics["recall"]["mean"],
                "NDCG": metrics["ndcg"]["mean"],
                "Coverage": metrics["coverage"],
                "Diversity": metrics["diversity"]["mean"],
                "Users": metrics["n_users_evaluated"],
            }
        )
    return rows

comparison = pd.DataFrame(
    flatten_results("Baseline", baseline_results) + flatten_results("Hybrid", hybrid_results)
)
comparison["Model"] = pd.Categorical(comparison["Model"], ["Baseline", "Hybrid"], ordered=True)
comparison = comparison.sort_values(["K", "Model"]).reset_index(drop=True)

display(comparison)

print("Model    | K  | Precision | Recall | NDCG  | Coverage | Diversity")
print("---------|----|-----------|--------|-------|----------|----------")
for row in comparison.itertuples(index=False):
    print(
        f"{row.Model:<8} | {row.K:<2} | {row.Precision:.3f}     | {row.Recall:.3f}  | "
        f"{row.NDCG:.3f} | {row.Coverage:.3f}    | {row.Diversity:.3f}"
    )

,Model,K,Precision,Recall,NDCG,Coverage,Diversity,Users
0,Baseline,5,0.018729,0.015463,0.021310,0.010,0.900000,299
1,Hybrid,5,0.058863,0.044107,0.052674,0.638,0.300669,299
2,Baseline,10,0.017057,0.027046,0.024902,0.020,0.800000,299
3,Hybrid,10,0.053846,0.082016,0.064705,0.894,0.352657,299
4,Baseline,20,0.018227,0.059615,0.039289,0.040,0.857895,299
5,Hybrid,20,0.043144,0.132582,0.087289,0.936,0.407921,299


Model    | K  | Precision | Recall | NDCG  | Coverage | Diversity
---------|----|-----------|--------|-------|----------|----------
Baseline | 5  | 0.019     | 0.015  | 0.021 | 0.010    | 0.900
Hybrid   | 5  | 0.059     | 0.044  | 0.053 | 0.638    | 0.301
Baseline | 10 | 0.017     | 0.027  | 0.025 | 0.020    | 0.800
Hybrid   | 10 | 0.054     | 0.082  | 0.065 | 0.894    | 0.353
Baseline | 20 | 0.018     | 0.060  | 0.039 | 0.040    | 0.858
Hybrid   | 20 | 0.043     | 0.133  | 0.087 | 0.936    | 0.408


The comparison answers RQ1 by showing that the hybrid model improves ranking relevance over the non-personalized recency baseline in the Nepali e-commerce context. The baseline repeatedly recommends recent high-activity products, while the hybrid model combines collaborative behavior with product metadata, freshness, and festival-aware category signals. This raises NDCG and recall at larger K values, indicating that the hybrid approach is better at placing relevant products in users' recommendation lists while also surfacing more of the catalog.